# NLP Pipeline - Practice

## Installer les prérequis

In [11]:
!pip install requests beautifulsoup4 selenium webdriver-manager PyPDF2 spacy nltk
!python -m spacy download fr_core_news_sm
!pip install -q pdfplumber reportlab arabic-reshaper python-bidi transformers sentencepiece torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 59.1 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Devoir 

- Extraire les données depuis un pdf
- Utiliser des textes arabes au lieu de francais

In [12]:
import re
import sys
import subprocess
import unicodedata
from pathlib import Path
from collections import Counter
import pdfplumber


def ensure_pdf_dependencies():
    """Ensure dependencies for Arabic RTL PDF generation are available."""
    packages = {
        "reportlab": "reportlab",
        "arabic_reshaper": "arabic-reshaper",
        "bidi": "python-bidi",
    }
    for module_name, package_name in packages.items():
        try:
            __import__(module_name)
        except ImportError:
            print(f"Installing {package_name}...")
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--break-system-packages",
                package_name,
            ])


ensure_pdf_dependencies()

from bidi.algorithm import get_display
import arabic_reshaper
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas


def get_arabic_font_path():
    """Finds an available system font that supports Arabic or RTL text."""
    # Primary candidates (prefer dedicated Arabic fonts)
    primary_candidates = [
        "/usr/share/fonts/noto/NotoNaskhArabic-Regular.ttf",
        "/usr/share/fonts/opentype/noto/NotoNaskhArabic-Regular.otf",
        "/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf",
    ]
    
    # Secondary candidates (universal fonts with good Unicode support)
    secondary_candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/TTF/DejaVuSans.ttf",
        "/usr/share/fonts/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf",
        "/usr/share/fonts/opentype/unifont/unifont.otf",
    ]
    
    # Fallback for other systems
    tertiary_candidates = [
        "/System/Library/Fonts/Arial.ttf",  # macOS
        "C:\\Windows\\Fonts\\arial.ttf",  # Windows
    ]
    
    all_candidates = primary_candidates + secondary_candidates + tertiary_candidates
    
    for path in all_candidates:
        if Path(path).exists():
            print(f"✓ Using font: {path}")
            return path
    
    # If no hardcoded path works, search directories
    font_dirs = [
        Path("/usr/share/fonts/truetype"),
        Path("/usr/share/fonts/opentype"),
        Path("/usr/share/fonts"),
    ]
    
    for font_dir in font_dirs:
        if font_dir.exists():
            for ext in ["*.ttf", "*.otf"]:
                fonts = sorted(list(font_dir.glob(f"**/{ext}")))
                if fonts:
                    selected = str(fonts[0])
                    print(f"✓ Using fallback font: {selected}")
                    return selected
    
    raise FileNotFoundError(
        "No suitable fonts found on system. "
        "Please install fonts-dejavu or fonts-noto-naskh-arabic."
    )


def shape_rtl_for_pdf(text):
    """Converts Arabic text to displayable RTL form for ReportLab."""
    try:
        reshaped = arabic_reshaper.reshape(text)
        return get_display(reshaped)
    except Exception as e:
        print(f"Warning: RTL shaping failed: {e}. Using original text.")
        return text


def create_sample_pdf(pdf_path, lines):
    """Creates a PDF with Arabic RTL rendering."""
    font_path = get_arabic_font_path()
    font_name = "ArabicFont"

    if font_name not in pdfmetrics.getRegisteredFontNames():
        pdfmetrics.registerFont(TTFont(font_name, font_path))

    c = canvas.Canvas(str(pdf_path), pagesize=A4)
    width, height = A4
    y = height - 60
    c.setFont(font_name, 14)

    for line in lines:
        rtl_line = shape_rtl_for_pdf(line)
        try:
            c.drawRightString(width - 40, y, rtl_line)
        except Exception as e:
            print(f"Warning: Could not render line. Trying left-aligned: {e}")
            c.drawString(40, y, rtl_line)
        y -= 32
        if y < 60:
            c.showPage()
            c.setFont(font_name, 14)
            y = height - 60

    c.save()
    print(f"✓ PDF created: {pdf_path}")


def extract_pdf_text(path):
    """Extracts text from PDF using pdfplumber."""
    chunks = []
    try:
        with pdfplumber.open(path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    chunks.append(text)
        result = "\n".join(chunks)
        print(f"✓ Extracted {len(chunks)} page(s) from PDF")
        return result
    except Exception as e:
        print(f"Warning: PDF extraction failed: {e}")
        return ""


ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652]")
NON_ARABIC = re.compile(r"[^\u0600-\u06FF\s]")
MULTI_SPACE = re.compile(r"\s+")


def normalize_arabic(text):
    """Normalizes Arabic text: removes diacritics, normalizes variants."""
    text = unicodedata.normalize("NFKC", text)
    text = ARABIC_DIACRITICS.sub("", text)
    text = text.replace("ـ", "")
    text = re.sub(r"[إأآا]", "ا", text)
    text = text.replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي").replace("ة", "ه")
    text = NON_ARABIC.sub(" ", text)
    text = MULTI_SPACE.sub(" ", text).strip()
    return text


def tokenize_arabic(text):
    """Tokenizes Arabic text by splitting on whitespace."""
    return [token for token in text.split() if len(token) > 1]


RAW_ARABIC_STOPWORDS = {
    "في", "من", "على", "إلى", "الى", "عن", "هذا", "هذه",
    "هو", "هي", "كما", "أن", "ان", "كان", "لقد", "ما", "لا", "مع", "ثم"
}
ARABIC_STOPWORDS = {normalize_arabic(word) for word in RAW_ARABIC_STOPWORDS}


def remove_stopwords(tokens):
    """Filters out common Arabic stopwords."""
    return [tok for tok in tokens if tok not in ARABIC_STOPWORDS]


def rtl_line_score(line):
    """Scores a line based on Arabic word patterns."""
    tokens = tokenize_arabic(normalize_arabic(line))
    score = sum(tok.startswith("ال") for tok in tokens)
    score += sum(tok in ARABIC_STOPWORDS for tok in tokens)
    return score


def restore_extracted_rtl_text(text):
    """Corrects line orientation if text extraction reversed any lines."""
    fixed_lines = []
    for line in text.splitlines():
        if re.search(r"[\u0600-\u06FF\uFB50-\uFDFF\uFE70-\uFEFF]", line):
            normal_score = rtl_line_score(line)
            reversed_score = rtl_line_score(line[::-1])
            fixed_lines.append(line[::-1] if reversed_score > normal_score else line)
        else:
            fixed_lines.append(line)
    return "\n".join(fixed_lines)


def arabic_pdf_pipeline_local():
    """Main pipeline: create PDF, extract, normalize, tokenize, and analyze Arabic text."""
    source_lines = [
        "يهدف هذا المشروع إلى استخراج النصوص العربية من ملفات بي دي اف وتحليلها باستخدام تقنيات الذكاء الاصطناعي.",
        "نقوم بتنظيف النصوص ومعالجتها ثم تحويلها إلى تمثيلات عددية باستخدام نموذج AraBERT.",
        "تساعد هذه الطريقة في فهم المحتوى النصي واستخدامه في تطبيقات مثل التصنيف وتحليل المشاعر.",
    ]

    pdf_path = Path.cwd() / "arabic.pdf"
    
    print("=" * 60)
    print("ARABIC PDF PROCESSING PIPELINE")
    print("=" * 60)
    
    # Step 1: Create PDF
    print("\n[1/5] Creating PDF with Arabic text...")
    create_sample_pdf(pdf_path, source_lines)

    # Step 2: Extract text
    print("\n[2/5] Extracting text from PDF...")
    extracted_raw = extract_pdf_text(pdf_path)
    
    # Step 3: Fix RTL orientation
    print("[3/5] Correcting RTL text orientation...")
    extracted_text = restore_extracted_rtl_text(extracted_raw)
    
    # Step 4: Normalize and tokenize
    print("[4/5] Normalizing Arabic and tokenizing...")
    normalized_text = normalize_arabic(extracted_text)
    tokens = tokenize_arabic(normalized_text)
    filtered_tokens = remove_stopwords(tokens)

    # Step 5: Fallback if extraction failed
    data_source = "pdf_extraction"
    if not filtered_tokens:
        print("⚠ No tokens extracted from PDF. Using source lines as fallback...")
        data_source = "fallback_source_lines"
        normalized_text = normalize_arabic(" ".join(source_lines))
        tokens = tokenize_arabic(normalized_text)
        filtered_tokens = remove_stopwords(tokens)

    frequencies = Counter(filtered_tokens)

    # Results
    print("\n[5/5] Pipeline complete!")
    print("\n" + "=" * 60)
    print("RESULTS")
    print("=" * 60)
    
    print(f"\n📄 PDF Path: {pdf_path}")
    print(f"📊 Data Source: {data_source}")
    print(f"📝 Tokens Found: {len(filtered_tokens)}")
    print(f"🔤 Unique Words: {len(frequencies)}")

    print("\n--- Sample (first 200 chars) ---")
    print(extracted_text[:200] + "..." if len(extracted_text) > 200 else extracted_text)

    print("\n--- Filtered Tokens (20 samples) ---")
    for i, token in enumerate(filtered_tokens[:20], 1):
        print(f"  {i:2d}. {token}")

    print("\n--- Top 10 Most Frequent Words ---")
    for rank, (word, count) in enumerate(frequencies.most_common(10), 1):
        bar = "█" * count
        print(f"  {rank:2d}. {word:<10} {bar} ({count})")

    print("\n" + "=" * 60)
    
    return {
        "pdf_path": str(pdf_path),
        "source": data_source,
        "extracted_raw": extracted_raw,
        "extracted_text": extracted_text,
        "tokens": filtered_tokens,
        "frequencies": frequencies,
    }


if __name__ == "__main__":
    try:
        pipeline_result = arabic_pdf_pipeline_local()
        print("\n✓ Pipeline executed successfully!")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

ARABIC PDF PROCESSING PIPELINE

[1/5] Creating PDF with Arabic text...
✓ Using fallback font: /usr/share/fonts/truetype/droid/DroidSansFallbackFull.ttf
✓ PDF created: /kaggle/working/arabic.pdf

[2/5] Extracting text from PDF...
✓ Extracted 1 page(s) from PDF
[3/5] Correcting RTL text orientation...
[4/5] Normalizing Arabic and tokenizing...
⚠ No tokens extracted from PDF. Using source lines as fallback...

[5/5] Pipeline complete!

RESULTS

📄 PDF Path: /kaggle/working/arabic.pdf
📊 Data Source: fallback_source_lines
📝 Tokens Found: 34
🔤 Unique Words: 32

--- Sample (first 200 chars) ---
                                                                                                       
                                                                                
               ...

--- Filtered Tokens (20 samples) ---
   1. يهدف
   2. المشروع
   3. استخراج
   4. النصوص
   5. العربيه
   6. ملفات
   7. بي
   8. دي
   9. اف
  10. وتحليلها
  11. باستخدام
  12. تقنيات
  13. الذكاء
  1